In [ ]:
from pathlib import Path
import glob
import shlex
import subprocess

import deeplabcut
import pandas as pd
from IPython.display import Video, display


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
project_name = 'FishPoseProject'
experimenter = 'Lucas'
working_directory = PROJECT_ROOT / 'DeepLabCut'
config_path = str(working_directory / f'{project_name}-{experimenter}' / 'config.yaml')

hr_root = PROJECT_ROOT / 'data/raw/yoshidak/heart_rate'
script_path = PROJECT_ROOT / 'scripts/heart_rate/extract_heart_rate.py'
outdir = PROJECT_ROOT / 'results/heart_rate/hr_results_v2'

videos = sorted(hr_root.rglob('*.MOV'))
video_paths = [str(v) for v in videos]
video_paths[:5], len(video_paths)


In [ ]:
# Generate DeepLabCut sidecars for the heart-rate videos when needed.
# Skip this cell if the DLC CSV/H5 files are already present.
deeplabcut.analyze_videos(config_path, video_paths, batchsize=64, save_as_csv=True)

In [ ]:
def find_track_sidecars(video_path: Path):
    stem = video_path.stem
    sidecars = sorted(video_path.parent.glob(f'{stem}*DLC*.csv'))
    sidecars += sorted(video_path.parent.glob(f'{stem}*DLC*.h5'))
    sidecars += sorted(video_path.parent.glob(f'{stem}*DeepCut*.csv'))
    sidecars += sorted(video_path.parent.glob(f'{stem}*DeepCut*.h5'))
    return sorted({p for p in sidecars})

track_status = pd.DataFrame(
    {
        'video': [v.name for v in videos],
        'track_files': [', '.join(p.name for p in find_track_sidecars(v)) for v in videos],
        'has_tracks': [len(find_track_sidecars(v)) > 0 for v in videos],
    }
)
track_status

In [ ]:
missing = track_status.loc[~track_status['has_tracks'], 'video'].tolist()
if missing:
    raise RuntimeError(f'Missing DLC outputs for: {missing}')
track_status

In [ ]:
single_video = videos[0]
single_cmd = [
    'python',
    str(script_path),
    '--root', str(single_video.parent),
    '--track_file', str(find_track_sidecars(single_video)[0]),
    '--roi_mode', 'tracked_body_patch',
    '--outdir', str(outdir / single_video.stem),
    '--include_free_swim',
    '--track_pcutoff', '0.6',
    '--max_gap_frames', '5',
    '--min_valid_fraction', '0.75',
]
print(' '.join(shlex.quote(part) for part in single_cmd))
subprocess.run(single_cmd, check=True)

In [ ]:
batch_cmd = [
    'python',
    str(script_path),
    '--root', str(hr_root),
    '--roi_mode', 'tracked_body_patch',
    '--outdir', str(outdir),
    '--include_free_swim',
    '--track_pcutoff', '0.6',
    '--max_gap_frames', '5',
    '--min_valid_fraction', '0.75',
]
print(' '.join(shlex.quote(part) for part in batch_cmd))
subprocess.run(batch_cmd, check=True)

In [ ]:
summary_csv = outdir / 'moving_window_multiROI_summary.csv'
summary_df = pd.read_csv(summary_csv)
summary_df.head()

In [ ]:
overlay_candidates = sorted(outdir.rglob('*tracked_body_patch_overlay.mp4'))
qc_candidates = sorted(outdir.rglob('*tracked_body_patch_qc.csv'))
overlay_candidates[:3], qc_candidates[:3]

In [ ]:
if overlay_candidates:
    display(Video(str(overlay_candidates[0]), embed=False))
if qc_candidates:
    display(pd.read_csv(qc_candidates[0]).head())
